# 035-loss-arms-rerun: analysis

# Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.sequence import village_bootstrap

# Params

In [ ]:
RUNS = {"control": ("035.1", "033.1"), "building": ("035.2", "033.2"), "structure": ("035.3", "033.3")}   # arm: (rerun, source)
CONTROL = "control"
LOSS_ARM = "building"
REPRODUCED_TOLERANCE_M = 0.01
BOOTSTRAP_RESAMPLES = 10_000
SEED = 42
ARTIFACTS = Path("artifacts")
SOURCE_ARTIFACTS = Path("../033-class-loss-in-system/artifacts")
FIGURES = Path("figures")
KEYS = ["village", "route_index"]

# Data

In [ ]:
rerun = {arm: pd.read_csv(ARTIFACTS / f"flights-{new}.csv") for arm, (new, _) in RUNS.items()
         if (ARTIFACTS / f"flights-{new}.csv").exists()}
source = {arm: pd.read_csv(SOURCE_ARTIFACTS / f"flights-{old}.csv") for arm, (_, old) in RUNS.items()}
{arm: len(t) for arm, t in rerun.items()}

# Model

In [ ]:
def flight_gap_m(a: pd.DataFrame, b: pd.DataFrame) -> pd.Series:
    """Per-flight end-error difference a - b, paired by (village, route); raises if the flight sets differ."""
    paired = a[KEYS + ["end_error_m"]].merge(b[KEYS + ["end_error_m"]], on=KEYS, suffixes=("_a", "_b"), validate="1:1")
    if len(paired) != len(a) or len(paired) != len(b):
        raise ValueError(f"flight sets differ: {len(a)} / {len(b)} / {len(paired)} paired")
    return (paired["end_error_m_a"] - paired["end_error_m_b"]).set_axis(pd.MultiIndex.from_frame(paired[KEYS]))

# Run

In [ ]:
summary = pd.DataFrame({arm: {"run_035": RUNS[arm][0], "run_033": RUNS[arm][1],
                               "mean_035_m": rerun[arm]["end_error_m"].mean(), "mean_033_m": source[arm]["end_error_m"].mean(),
                               "median_035_m": rerun[arm]["end_error_m"].median(), "median_033_m": source[arm]["end_error_m"].median(),
                               "max_abs_flight_gap_m": flight_gap_m(rerun[arm], source[arm]).abs().max(),
                               "flights_differing": int((flight_gap_m(rerun[arm], source[arm]).abs() > REPRODUCED_TOLERANCE_M).sum())}
                         for arm in rerun}).T
summary

In [ ]:
cross_match = pd.DataFrame({f"035 {new_arm}": {f"033 {old_arm}": flight_gap_m(rerun[new_arm], source[old_arm]).abs().max()
                                                for old_arm in source} for new_arm in rerun})
cross_match.round(3)

In [ ]:
bootstrap = pd.DataFrame([dict(zip(["delta_m", "ci_low", "ci_high"],
                                     village_bootstrap(tables[LOSS_ARM], tables[CONTROL], "end_error_m", BOOTSTRAP_RESAMPLES, SEED)),
                                 flights=name, a=LOSS_ARM, b=CONTROL)
                          for name, tables in (("035", rerun), ("033", source)) if LOSS_ARM in tables and CONTROL in tables])
bootstrap

In [ ]:
results = {"summary": summary.to_dict("index"), "cross_match_max_abs_m": cross_match.to_dict(),
           "bootstrap": bootstrap.to_dict("records")}
(ARTIFACTS / "analysis.json").write_text(json.dumps(results, indent=1, default=float))

# Plots

In [ ]:
fig, axes = plt.subplots(1, len(rerun), figsize=(4 * len(rerun), 4), squeeze=False)
for ax, arm in zip(axes[0], rerun):
    paired = rerun[arm][KEYS + ["end_error_m"]].merge(source[arm][KEYS + ["end_error_m"]], on=KEYS, suffixes=("_035", "_033"))
    ax.loglog(paired["end_error_m_033"], paired["end_error_m_035"], "o", ms=3)
    ax.plot([0.1, 300], [0.1, 300], "k--", lw=0.8)
    ax.set_title(f"{arm}: {RUNS[arm][0]} vs {RUNS[arm][1]}", fontsize=9)
    ax.set_xlabel("033 end error, m"); ax.set_ylabel("035 end error, m")
fig.tight_layout()
FIGURES.mkdir(exist_ok=True)
fig.savefig(FIGURES / "rerun-vs-033.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({f"{arm} {RUNS[arm][0]}": t["end_error_m"] for arm, t in rerun.items()}, ax)
ax.set_title("035: end error on the 94 test flights, re-run from 033 artifacts")
fig.tight_layout()
fig.savefig(FIGURES / "end-error-cdf.png", dpi=150)